In [1]:
import warnings

# Ẩn tất cả warning
warnings.filterwarnings("ignore")

# Hoặc chỉ ẩn riêng FutureWarning / DeprecationWarning
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [2]:
from collections import defaultdict, Counter
from tqdm import tqdm
import pandas as pd
import numpy as np
import re
import unicodedata

def preprocess_text(text):
    """
    Hàm preprocess text: làm sạch và chuẩn hóa text từ trang PDF.
    """
    # Chuẩn hóa tiếng Việt
    text = unicodedata.normalize('NFC', text)
    
    # Thay thế tab (\t) thành space
    text = text.replace('\t', ' ')
    
    # Giữ nguyên ký tự tiếng Việt, chỉ xóa ký tự đặc biệt không cần thiết
    text = re.sub(r'[^\w\sÀ-ỹ.,!?]', ' ', text)  
    
    # Xóa xuống dòng, khoảng trắng thừa
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    
    # Xử lý hyphen (nối từ bị ngắt dòng)
    text = re.sub(r'-\s*\n\s*', '', text)
    
    # Xóa khoảng trắng trước dấu câu
    text = re.sub(r'\s([?.!,;:])', r'\1', text)

    # Xóa số trang (dài hơn 2 chữ số) hoặc số đứng 1 mình
    text = re.sub(r'\b\d{3,}\b', ' ', text)  
    
    # Xóa các chuỗi toàn số hoặc số đứng lẻ
    text = re.sub(r'\b\d+\b', '', text)

    # Xóa khoảng trắng thừa
    text = re.sub(r'\s+', ' ', text).strip()

    # Xóa số index đầu dòng (1., 2., 3., 4, ...)
    text = re.sub(r'^\d+\.\s*|\b\d+\s', ' ', text)  
    
    # Xóa các từ như "CHƯƠNG", "Phần", "bảng" (bao gồm biến thể viết hoa/thường)
    text = re.sub(r'\b(CHƯƠNG|Phần|bảng|CHƯƠNG|PHẦN|BẢNG)\b', ' ', text, flags=re.IGNORECASE)
    
    # Strip lại lần cuối
    text = text.strip()

    return text

In [3]:
!pip install py_vncorenlp

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.6 MB/s eta 0:00:0000:0100:01
  Created wheel for py_vncorenlp: filename=py_vncorenlp-0.1.4-py3-none-any.whl size=4304 sha256=6a58fc4a2de340b26775cd8d0e49cb8f913a743fc58a064d3a24bd92d7b1e42a
  Stored in directory: /root/.cache/pip/wheels/6d/2d/d6/158260bfd6820d144535857b80cc112bee5c3aa6d81b6dc049
Successfully built py_vncorenlp


In [4]:
from py_vncorenlp import VnCoreNLP

# Khởi tạo model 
model_vncorenlp = VnCoreNLP(annotators=["wseg", "pos", "ner", "parse"],
                  save_dir="/kaggle/input/vncorenlp/models/vncorenlp")  # sửa path nếu cần

2025-09-01 01:44:55 INFO  WordSegmenter:24 - Loading Word Segmentation model
2025-09-01 01:44:55 INFO  PosTagger:23 - Loading POS Tagging model
2025-09-01 01:44:58 INFO  NerRecognizer:34 - Loading NER model
2025-09-01 01:45:12 INFO  DependencyParser:32 - Loading Dependency Parsing model


In [5]:
class TextPreprocessor:
    def __init__(self, model):
        """
        Khởi tạo bộ tiền xử lý văn bản sử dụng VnCoreNLP.
        :param model: Đối tượng VnCoreNLP đã được tải trước đó.
        """
        self.model = model  # Dùng model đã tải

    def preprocess(self, text):
        """
        Tiền xử lý văn bản với cả 4 annotators trên init.

        return: Dictionary chứa các thông tin:
                - 'word_segmented': Chuỗi đã được tách từ
                - 'pos_tags': Danh sách từ và từ loại tương ứng
                - 'ner_tags': Danh sách từ và thực thể có tên
                - 'parse_tree': Cấu trúc phân tích cú pháp (cấu trúc cây bắt đầu bằng ROOT)
        """
        output = self.model.annotate_text(text)

        word_segmented = []
        # pos_tags = []
        # ner_tags = []
        # parse_tree = []

        # Kiểm tra xem output có đúng định dạng không
        if not isinstance(output, dict):
            raise ValueError(f"Unexpected output format: {type(output)}")

        for sentence_id, words in output.items():  # Lặp qua từng câu
            if isinstance(words, list) and all(isinstance(word, dict) for word in words):
                # Xử lý tách từ
                word_segmented.extend([word["wordForm"] for word in words])

                # Xử lý POS (Part-of-Speech)
                #pos_tags.extend([(word["wordForm"], word["posTag"]) for word in words])

                # Xử lý NER (Named Entity Recognition)
                #ner_tags.extend([(word["wordForm"], word["nerLabel"]) for word in words if word["nerLabel"] != "O"])

                # Xử lý Parse Tree (Dependency Parsing)
                #parse_tree.append([(word["wordForm"], word["head"], word["depLabel"]) for word in words])
            else:
                print(f"Unexpected sentence format: {words}")

        return {
            "word_segmented_join": " ".join(word_segmented),
            "word_segmented": word_segmented
            #"pos_tags": pos_tags,
            #"ner_tags": ner_tags,
            #"parse_tree": parse_tree
        }

In [6]:
from typing import List

class DataPreprocessor:
    def __init__(self):
        """
        Initialize the DataPreprocessor with the TextPreprocessor using VnCoreNLP.
        """
        self.text_preprocessor = TextPreprocessor(model_vncorenlp)

    def _tokenize(self, text: str) -> List[str]:
        """Convert text to token list using VnCoreNLP via TextPreprocessor."""
        return self.text_preprocessor.preprocess(text)["word_segmented"]

    def tokenize_and_count(self, df: pd.DataFrame, text_col: str = "chunk") -> pd.DataFrame:
        """
        Tokenize each row in the given text_col and count tokens.
        Adds a new column 'token_count'.
        """
        token_counts = []
        for _, row in df.iterrows():
            text = str(row[text_col])
            tokens = self._tokenize(text)
            token_counts.append(len(tokens))
        df = df.copy()
        df['token_count'] = token_counts
        return df

    def tokenize_all(self, df: pd.DataFrame, text_col: str = "chunk") -> pd.DataFrame:
        """
        Tokenize all text in the DataFrame and add columns with token lists and annotations.
        Returns a new DataFrame with 'word_segmented_join', 'tokens'.
        """
        df_copy = df.copy()
        df_copy[['word_segmented_join', 'tokens']] = df_copy[text_col].apply(
            lambda x: pd.Series(self.text_preprocessor.preprocess(str(x)))
        )
        return df_copy

In [7]:
!pip install FlagEmbedding

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 3.3 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 88.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 62.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 29.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.8 MB/s eta 0:00

In [8]:
import torch
from FlagEmbedding import BGEM3FlagModel
class EmbeddingGenerator:
    def __init__(self, model_name="BAAI/bge-m3", device=None, max_length=512, vncorenlp = None):
        """
        Khởi tạo EmbeddingGenerator với mô hình BGE-M3
        
        Args:
            model_name: Tên mô hình embedding (mặc định: BAAI/bge-m3)
            device: Thiết bị để chạy mô hình ('cpu' hoặc 'cuda')
            max_length: Độ dài tối đa của văn bản đầu vào
        """
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        
        self.device = device
        self.model_name = model_name
        self.max_length = max_length
        self.vncorenlp = vncorenlp
        
        # Khởi tạo mô hình
        self.model = BGEM3FlagModel(model_name, device=device)

    def embed_query(self, processed_text):
        """
        Tính embedding cho một câu truy vấn sau khi tiền xử lý
        
        Args:
            processed_text: Câu truy vấn cần tính embedding đã được xử lý
            
        Returns:
            Embedding vector cho câu truy vấn (dense vector)
        """
        try:
            # Kiểm tra đầu vào
            if not processed_text or not isinstance(processed_text, str):
                raise ValueError("Processed text phải là một chuỗi không rỗng.")
            
            # Tính embedding
            embedding = self.model.encode(processed_text)
            #print("Raw embedding:", embedding)  # Debug đầu ra thô
            
            # Lấy dense vector từ embedding
            if isinstance(embedding, dict) and 'dense_vecs' in embedding:
                return embedding['dense_vecs'].tolist()  # Trả về toàn bộ vector
            else:
                raise ValueError("Embedding không chứa 'dense_vecs'.")
        except Exception as e:
            print(f"Lỗi khi tính embedding cho query: {e}")
            return [0.0] * 1024  # Trả về vector rỗng nếu lỗi

    def embed_documents(self, texts):
        """
        Tính embedding cho danh sách các văn bản
        
        Args:
            texts: Danh sách các văn bản
            
        Returns:
            List các embedding vector
        """
        # Tính embedding trực tiếp cho toàn bộ danh sách
        embeddings = self.model.encode(texts) # Dùng word_segment để lấy mỗi dense vector
        if isinstance(embeddings, dict) and 'dense_vecs' in embeddings:
            return [v.tolist() for v in embeddings['dense_vecs']]
        return [v.tolist() for v in embeddings]

    def preprocess_and_tokenize(self, text):
        """
        Tiền xử lý và tokenize văn bản sử dụng VnCoreNLP
        
        Args:
            text: Văn bản đầu vào
            
        Returns:
            Chuỗi token đã được tách từ
        """
        try:
            if self.vncorenlp is None:
                return text  # Trả về văn bản gốc nếu không có VnCoreNLP
            # Chuẩn hóa văn bản
            text = text.strip().lower()  # Chuyển về chữ thường và loại bỏ khoảng trắng thừa
            # Tách từ bằng VnCoreNLP
            annotated = self.vncorenlp.annotate_text(text)
            # Lấy danh sách từ (sử dụng 'wordForm') và nối không có _
            tokenized = [token['wordForm'] for token in annotated[0]]
            # In danh sách token
            print("Tokenized text:", tokenized)  # Kiểm tra kết quả tokenization
            # Loại bỏ khoảng trắng thừa và nối lại thành chuỗi
            tokenized = [word for word in tokenized if word]  # Loại bỏ từ rỗng
            tokenized = [re.sub(r'\s+', ' ', word) for word in tokenized]  # Loại bỏ khoảng trắng thừa trong từ
            tokenized = [word.strip() for word in tokenized]  # Loại bỏ khoảng trắng đầu và cuối
            # Trả về chuỗi token đã được nối lại
            return " ".join(tokenized)  # Nối lại thành chuỗi với khoảng trắng thông thường
        except Exception as e:
            print(f"Lỗi khi tiền xử lý và tokenize: {e}")
            return text  # Trả về văn bản gốc nếu lỗi
        
    def get_dense_size(self):
        """
        Lấy kích thước của dense vector bằng cách tạo một data test để tính size
        
        Returns:
            Kích thước của dense vector
        """
        sample_text = "This is a test sentence."
        sample_embedding = self.model.encode(sample_text)
        if isinstance(sample_embedding, dict) and 'dense_vecs' in sample_embedding:
            return len(sample_embedding['dense_vecs'][0])
        return len(sample_embedding[0])

2025-09-01 01:47:04.672595: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756691225.057801      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756691225.169432      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [9]:
# Khởi tạo preprocessor
preprocessor = DataPreprocessor()

In [10]:
# Khởi tạo EmbeddingGenerator
embedding_generator = EmbeddingGenerator(vncorenlp=model_vncorenlp)

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

...

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

sparse_linear.pt:   0%|          | 0.00/3.52k [00:00<?, ?B/s]

In [11]:
!pip install qdrant_client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.3/337.3 kB 5.5 MB/s eta 0:00:00a 0:00:01


In [12]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

In [13]:
import os
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")
GROQ_API_KEY_2 = user_secrets.get_secret("GROQ_API_KEY_2")
GROQ_API_KEY_3 = user_secrets.get_secret("GROQ_API_KEY_3")
QDRANT_API_KEY = user_secrets.get_secret("QDRANT_API_KEY")
GPT = user_secrets.get_secret("GPT")
GG_API_KEY = user_secrets.get_secret("GG_API_KEY")

In [14]:
# Khởi tạo client Qdrant
qdrant_client = QdrantClient(url='https://72249cbf-3fe1-4881-b625-d4f1cb122aea.europe-west3-0.gcp.cloud.qdrant.io',
                             api_key=QDRANT_API_KEY, timeout=500)  # Sửa URL và API_KEY nếu cần

# Tạo collection trong Qdrant
collection_name = "test_llm"

In [15]:
from FlagEmbedding import BGEM3FlagModel, FlagReranker
   
# Khởi tạo mô hình reranking ViRanker
def load_viranker(device = None):
    try:
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
            
        reranker = FlagReranker('namdp-ptit/ViRanker', use_fp16=True, device = device)
        return reranker
    except Exception as e:
        print(f"Lỗi khi tải mô hình ViRanker: {e}")
        raise

def rerank_viranker(query, documents, reranker, top_k=1, normalize=True):
    try:
        # Tạo danh sách cặp (query, document)
        pairs = [[query, doc] for doc in documents]
        # Tính score
        scores = reranker.compute_score(pairs, normalize=normalize, batch_size=5)
        ranked_pairs = sorted(zip(scores, documents), reverse=True)
        return ranked_pairs[:top_k]
    except Exception as e:
        print(f"Lỗi khi rerank: {e}")
        return list(zip([0.0] * len(documents), documents))[:top_k]

In [16]:
!pip install langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.4/131.4 kB 3.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.0/444.0 kB 14.4 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.66
    Uninstalling langchain-core-0.3.66:
      Successfully uninstalled langchain-core-0.3.66


In [17]:
!pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 0.3.8
    Uninstalling langchain-text-splitters-0.3.8:
      Successfully uninstalled langchain-text-splitters-0.3.8
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.26
    Uninstalling langchain-0.3.26:
      Successfully uninstalled langchain-0.3.26
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is t

In [18]:
!pip install bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.9 MB/s eta 0:00:00


In [19]:
!pip install keybert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 1.6 MB/s eta 0:00:00


In [20]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.callbacks import get_openai_callback
import os
import time
from bert_score import score as bert_score
import json

In [21]:
from functools import lru_cache
from kaggle_secrets import UserSecretsClient
from langchain_core.messages import HumanMessage
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
from sklearn.feature_extraction.text import TfidfVectorizer
from tenacity import retry, stop_after_attempt, wait_exponential
import logging
from tqdm import tqdm
from keybert import KeyBERT

In [22]:
# Biến toàn cục để lưu LLM
_llm = None

In [23]:
@lru_cache(maxsize=1)
def load_llm():
    """Tải và lưu trữ LLM để tái sử dụng."""
    global _llm
    if _llm is None:
        try:
            _llm = ChatGroq(
                model="llama-3.1-8b-instant",
                temperature=0.1,
                max_tokens=200,
                api_key=GROQ_API_KEY_3
            )
            logging.info("LLM loaded successfully.")
        except Exception as e:
            logging.error(f"Error loading LLM: {e}")
            raise
    return _llm

In [24]:
import re

def normalize_text(text):
    """Chuẩn hóa văn bản để so sánh Disease vs Prediction."""
    text = str(text).lower().strip()       # về lowercase
    text = text.replace("_", " ")          # bỏ gạch dưới vncorenlp
    text = re.sub(r'[^\w\s]', ' ', text)   # bỏ ký tự đặc biệt
    text = re.sub(r'\s+', ' ', text)       # gom nhiều space thành 1
    return text.strip()

In [25]:
def generate_ngrams(text, max_n=None):
    """Sinh tất cả n-grams từ chuỗi text (1 đến max_n)."""
    tokens = normalize_text(text).split()
    if not max_n:
        max_n = len(tokens)
    ngrams = []
    for n in range(1, max_n + 1):
        for i in range(len(tokens) - n + 1):
            ngrams.append(" ".join(tokens[i:i+n]))
    return list(set(ngrams))  # loại bỏ trùng lặp

In [26]:
from typing import List
def extract_diseases(prediction: str, max_n: int = 4) -> List[List[str]]:
    """Tách danh sách bệnh từ câu trả lời, tạo n-gram cho từng bệnh."""
    try:
        # Tách theo dấu phẩy, bỏ khoảng trắng thừa
        diseases = [d.strip() for d in prediction.split(",") if d.strip()]
        # Bỏ từ "bệnh" và khoảng trắng
        diseases = [re.sub(r'\bbệnh\b', '', d, flags=re.IGNORECASE).strip() for d in diseases]
        # Tạo n-gram cho từng bệnh
        disease_ngrams = [generate_ngrams(disease, max_n=max_n) for disease in diseases if disease]
        return disease_ngrams  # Trả về danh sách các danh sách n-gram
    except Exception:
        return []

In [60]:
# Con evaluator khác (ko phải llama-3.1-8b)
eval_llm = ChatGroq(
    model="gemma2-9b-it", 
    temperature=0,
    api_key = GROQ_API_KEY_3,
    max_tokens = 10
)

In [61]:
def exact_match(disease: str, response: str) -> int:
    # Chuẩn hóa văn bản
    ans = normalize_text(disease)
    response_diseases = [r.strip() for r in response.split(',') if r.strip()]
   
    prompt = f"""
        Bạn là một chuyên gia y tế. Nhiệm vụ của bạn là kiểm tra xem
        có bất kỳ phần tử nào trong danh sách 'Response' giống với 'Ground Truth Disease' hay không và trả lời 0 hoặc 1
        - So sánh KHÔNG PHÂN BIỆT CHỮ HOA/THƯỜNG, KHÔNG PHÂN BIỆT DẤU CÁCH, và KHÔNG PHÂN BIỆT DẤU TIẾNG VIỆT.
        - DUYỆT QUA TỪNG PHẦN TỬ trong danh sách 'Response' và so sánh với 'Ground Truth Disease'.
        - Nếu CÓ ÍT NHẤT MỘT PHẦN TỬ trong 'Response' giống 'Ground Truth Disease', BẮT BUỘC trả lời "1".
        - Nếu KHÔNG CÓ PHẦN TỬ NÀO giống hoặc không đủ thông tin, BẮT BUỘC trả lời "0".
        - CHỈ ĐƯỢC TRẢ LỜI "1" HOẶC "0", KHÔNG ĐƯỢC TRẢ LỜI SỐ THẬP PHÂN, ĐIỂM SỐ, HOẶC BẤT KỲ GIẢI THÍCH NÀO.
        - Ví dụ:
          Ground Truth Disease: gout & Response: [GOUT, nhức chân] => "1"
          Ground Truth Disease: nhức chân & Response: [gout, mỏi người] => "0"
          Ground Truth Disease: bàn chân đái tháo đường & Response: [Bàn Chân Đái Tháo Đường, nhức chân] => "1"
          Ground Truth Disease: Béo phì & Response: [béo phì cấp độ 1, gout] => "1"
          Ground Truth Disease: tiểu đường & Response: [nhức chân, gout] => "0"

        Ground Truth Disease: {ans}
        Response: {response_diseases} (danh sách các bệnh tách từ response bằng dấu phẩy)

        
        """
    try:
        response = eval_llm.invoke(prompt)
        llm_response = response.content.strip()  # Lấy nội dung phản hồi
        #output_tokens = response.usage.completion_tokens  # Lấy số token đầu ra
        
        print(f"Debug - raw LLM response: {llm_response}")
        #print(f"Debug - output tokens: {output_tokens}")
 
        # Xử lý đầu ra
        try:
            if llm_response == "1":
                return 1
            elif llm_response == "0":
                return 0
            else:
                score = float(llm_response)
                return 1 if score > 0.5 else 0
        except ValueError:
            return 0
    except Exception as e:
        print(f"Lỗi khi gọi mô hình: {e}")
        return 0

In [65]:
# # Test các ví dụ
# test_cases = [
#     ("gout", "GOUT, nhức chân, mỏi người"),  # Đúng, kỳ vọng 1
#     ("bàn chân đái tháo đường", "BÀN CHÂN ĐÁI THÁO ĐƯỜNG, nhức chân"),  # Đúng, kỳ vọng 1
#     ("béo phì", "béo phì cấp độ 1, gout"),  # Đúng, kỳ vọng 1
#     ("nhức chân", "gout, mỏi người"),  # Sai, kỳ vọng 0
#     ("tiểu đường", "nhức chân, gout, béo phì"),  # Sai, kỳ vọng 0
#     ("đau dạ dày", "nhức chân, mỏi người, gout"),  # Sai, kỳ vọng 0
#     ("viêm họng", "VIÊM HỌNG, sốt, ho"),  # Đúng, kỳ vọng 1
#     ("đau lưng", "đau lưng dưới, nhức chân"),  # Đúng, kỳ vọng 1
#     ("hen suyễn", "HEN SUYỄN, khó thở, gout"),  # Đúng, kỳ vọng 1
#     ("sốt xuất huyết", "nhức đầu, sốt, gout"),  # Sai, kỳ vọng 0
#     ("viêm gan", "viêm họng, nhức chân, mỏi người"),  # Sai, kỳ vọng 0
#     ("đau khớp", "đau lưng, nhức chân, gout")  # Sai, kỳ vọng 0
# ]

# for disease, pred in test_cases:
#     result = exact_match(disease, pred)
#     print(f"Kết quả EM cho {disease} và {pred}: {result}")

Debug - raw LLM response: 1
Kết quả EM cho gout và GOUT, nhức chân, mỏi người: 1
Debug - raw LLM response: 1
Kết quả EM cho bàn chân đái tháo đường và BÀN CHÂN ĐÁI THÁO ĐƯỜNG, nhức chân: 1
Debug - raw LLM response: 1
Kết quả EM cho béo phì và béo phì cấp độ 1, gout: 1
Debug - raw LLM response: 0
Kết quả EM cho nhức chân và gout, mỏi người: 0
Debug - raw LLM response: 0
Kết quả EM cho tiểu đường và nhức chân, gout, béo phì: 0
Debug - raw LLM response: 0
Kết quả EM cho đau dạ dày và nhức chân, mỏi người, gout: 0
Debug - raw LLM response: 1
Kết quả EM cho viêm họng và VIÊM HỌNG, sốt, ho: 1
Debug - raw LLM response: 1
Kết quả EM cho đau lưng và đau lưng dưới, nhức chân: 1
Debug - raw LLM response: 1
Kết quả EM cho hen suyễn và HEN SUYỄN, khó thở, gout: 1
Debug - raw LLM response: 0
Kết quả EM cho sốt xuất huyết và nhức đầu, sốt, gout: 0
Debug - raw LLM response: 0
Kết quả EM cho viêm gan và viêm họng, nhức chân, mỏi người: 0
Debug - raw LLM response: 0
Kết quả EM cho đau khớp và đau lưng, 

In [ ]:
import re
import logging
from tenacity import retry, stop_after_attempt, wait_exponential

# Khởi tạo client OpenAI với API key (bạn cần export OPENAI_API_KEY hoặc gán trực tiếp)


def calculate_triad(prediction, question, ground_truth, context):
    """
    Tính TRIaD Score dựa trên Relevance, Informativeness và Faithfulness.
    Dùng GPT-4o-mini để đánh giá, tách biệt với Llama.
    """
    # Kiểm tra kiểu dữ liệu đầu vào
    for var, name in [(prediction, "prediction"), (question, "question"), (ground_truth, "ground_truth"), (context, "context")]:
        if not isinstance(var, str):
            #print(f"[ERROR] Expected string for {name}, got {type(var)}: {var}")
            raise TypeError(f"Invalid type for {name}: {type(var)}")
    scores = {}
    aspects = {
        "Relevance": f"Câu hỏi: {question}\nTrả lời: {prediction}\n"
                     f"Trả lời có liên quan đến câu hỏi đến mức nào?\n"
                     f"Chỉ trả về duy nhất 1 số dạng float từ 0 đến 1. Không giải thích thêm",

        "Informativeness": f"Câu hỏi: {question}\nTrả lời: {prediction}\n"
                           f"Trả lời có cung cấp đủ thông tin (tên bệnh) để giải quyết câu hỏi không?\n"
                           f"Chỉ trả về duy nhất 1 số dạng float từ 0 đến 1. Không giải thích thêm",

        "Faithfulness": f"Ngữ cảnh: {context}\nTrả lời: {prediction}\n"
                        f"Trả lời có trung thực với thông tin trong ngữ cảnh không, "
                        f"hay có thêm thông tin ngoài ngữ cảnh?\n"
                        f"Chỉ trả về duy nhất 1 số dạng float từ 0 đến 1. Không giải thích thêm"
    }

    for aspect, prompt in aspects.items():
        try:
            response = eval_llm.invoke(prompt)  # Giả sử invoke là đồng bộ
            #print(f"[DEBUG] Type of response for {aspect}: {type(response)}")
            val = response.content
            #print(f"[DEBUG] {aspect} raw response: {val}")
            if not isinstance(val, str):
                #print(f"[ERROR] Expected string for response content, got {type(val)}: {val}")
                raise TypeError(f"Invalid type for response content: {type(val)}")
        except Exception as e:
            #print(f"[ERROR] Eval LLM invoke error for {aspect}: {e}")
            raise

        # Parse số
        match = re.findall(r"[-+]?\d*\.\d+", str(val))
        if not match:
            #print(f"[ERROR] Failed to parse number from output for {aspect}: {val}")
            raise ValueError(f"{aspect} không parse được số từ output: {val}")

        score = float(match[0])
        score = max(0.0, min(1.0, score))  # Clamp về [0,1]
        scores[aspect] = score

    triad_score = sum(scores.values()) / len(scores)
    return triad_score, scores


In [ ]:
from tenacity import retry, stop_after_attempt, wait_random_exponential

@retry(stop=stop_after_attempt(5), wait=wait_random_exponential(min=1, max=60))
def generate(query, embedding_generator, reranker, llm, enc=None):
    """
    Chatbot phân tích triệu chứng để xác định bệnh.
    Trả về: cleaned_answer, raw_answer, input_tokens, output_tokens, thời gian xử lý chi tiết.
    """
    try:
        # 1. Tạo embedding cho query
        start_time = time.time()
        query_embedding = embedding_generator.embed_query(query)
        embedding_time = time.time() - start_time

        # 2. Tìm kiếm trong Qdrant
        start_time = time.time()
        search_result = qdrant_client.search(
            collection_name=collection_name,
            query_vector=query_embedding,
            limit=5
        )
        documents = [result.payload["content"] for result in search_result]
        search_time = time.time() - start_time

        # 3. Rerank với ViRanker
        start_time = time.time()
        ranked_results = rerank_viranker(query, documents, reranker, top_k=1, normalize=True)
        context = "\n".join([f"{i+1}. {content}" for i, (_, content) in enumerate(ranked_results)])
        rerank_time = time.time() - start_time

        # 4. Prompt
        prompt = ChatPromptTemplate.from_messages([
            ("system", 
                "Bạn là bác sĩ AI, trả lời chỉ tên bệnh dựa trên triệu chứng. Liệt kê tối đa 4 bệnh, cách nhau bằng dấu phẩy, không giải thích."
            ),
            ("user", 
                f"Triệu chứng: {context}\nCâu hỏi: {query}\nTrả lời: "
            )
        ])

        # 5. Gọi LLM + đếm token
        start_time = time.time()
        with get_openai_callback() as cb:
            print("Call ....")
            chain = prompt | llm | StrOutputParser()
            raw_answer = chain.invoke({})
            in_tokens = cb.prompt_tokens
            out_tokens = cb.completion_tokens
        generate_time = time.time() - start_time

        # 6. Làm sạch output
        cleaned_answer = re.sub(r'^\d+\.\s*', '', raw_answer, flags=re.MULTILINE)
        cleaned_answer = re.sub(r'\s*\d+\.\s*', ' ', cleaned_answer)
        cleaned_answer = cleaned_answer.replace('\n', ' ').replace('.', '').strip()
        cleaned_answer = re.sub(r'\bbệnh\b', '', cleaned_answer, flags=re.IGNORECASE).strip()
        cleaned_answer = re.sub(r'\s+', ' ', cleaned_answer)

        total_time = embedding_time + search_time + rerank_time + generate_time

        logging.info(f"[Chatbot] Query: {query} | Answer: {cleaned_answer}")
        print("Complete generate!")
        return (
            cleaned_answer, raw_answer, in_tokens, out_tokens, context
        )

    except Exception as e:
        logging.error(f"Lỗi khi xử lý query '{query}': {e}")
        print("Fail generate!")
        return "", "", 0, 0, 0, 0, 0, 0, 0

In [ ]:
def process_batch(questions, llm, embedder, reranker, return_metrics=False):
    preds = []
    pred_embs = []
    metrics = []
    i = 1
    for question in questions:
        error, wrong_format = 0, 0
        try:
            # đo thời gian e2el
            e2el_start = time.time()

            # đo ttft
            ttft_start = time.time()
            answer, raw_answer, input_token, output_token, context = generate(question, embedder, reranker, llm)
            print(answer)
            ttft = time.time() - ttft_start
            print("Question: ",i)
            i +=1
            # embedding output (dùng cleaned_answer)
            pred_emb = embedder.embed_query(answer)
            print("embed already")
            # đo e2el
            e2el = time.time() - e2el_start
            if output_token > 35 or not answer:
                wrong_format = 1
            preds.append(answer)
            pred_embs.append(pred_emb)
            print("Append already!")
            if return_metrics:
                metrics.append({
                    "input_token": input_token,
                    "output_token": output_token,
                    "ttft": ttft,
                    "e2el": e2el,
                    "error": error,
                    "wrong_format": wrong_format,
                    "raw_answer": raw_answer,  # giữ lại raw để debug
                    "context": context
                })

        except Exception as e:
            error = 1
            preds.append("")
            print("Failed!")
            # pred_embs.append(np.zeros(embedder.get_sentence_embedding_dimension()))
            if return_metrics:
                metrics.append({
                    "input_token": 0,
                    "output_token": 0,
                    "ttft": 0,
                    "e2el": 0,
                    "error": error,
                    "wrong_format": 1,
                    "raw_answer": "",
                    "context":""
                })
    
    return (preds, np.array(pred_embs), metrics) if return_metrics else (preds)

In [ ]:
from sentence_transformers import SentenceTransformer, util as st_util
def evaluate_csv(csv_file, llm, embedder, reranker, batch_size=10):
    """
    Đánh giá file CSV với batch processing,
    gồm cả metric đánh giá (EM, BERT-Score, Cosine, TRIaD)
    và hiệu năng LLM (tokens, latency, error).
    """
    # Load dữ liệu
    df = pd.read_csv(csv_file)
    questions = df['Question'].tolist()
    ground_truths = df['Disease'].tolist()
    # Prediction metrics
    predictions, em_scores, bert_f1_scores, cosine_scores, triad_scores = [], [], [], [], []
    # Performance metrics
    all_input_tokens, all_output_tokens = [], []
    all_ttft, all_e2el = [], []
    all_errors, all_wrong_format = [], []
    raw_responses = [] # lưu raw output
    context = []
    # Loop theo batch
    for i in tqdm(range(0, len(questions), batch_size), desc="Processing batches"):
        batch_questions = questions[i:i + batch_size]
        batch_ground_truths = ground_truths[i:i + batch_size]
        # process_batch phải return prediction + embedding + metrics
        batch_preds, batch_pred_embs, batch_metrics = process_batch(
            batch_questions, llm, embedder, reranker, return_metrics=True
        )
        print("calculate...")
        # Lặp qua từng sample trong batch
        for idx, (pred, gt, pred_emb, m) in enumerate(zip(batch_preds, batch_ground_truths, batch_pred_embs, batch_metrics)):
            predictions.append(pred)
           
            # Lưu performance metrics
            all_input_tokens.append(m.get("input_token", 0))
            all_output_tokens.append(m.get("output_token", 0))
            all_ttft.append(m.get("ttft", 0))
            all_e2el.append(m.get("e2el", 0))
            all_errors.append(m.get("error", 0))
            all_wrong_format.append(m.get("wrong_format", 0))
            raw_responses.append(m.get("raw_answer", "")) # thêm raw respond
            context.append(m.get("context", ""))
            # EM (sử dụng hàm exact_match)
            em = exact_match(gt, pred) # Gọi hàm exact_match với ground truth và prediction
            em_scores.append(em)
            print('pred: ',pred)
            print('gt: ', gt)
           
            # BERT-Score F1
            try:
                _, _, f1 = bert_score([pred], [gt], lang="vi")
                bert_f1_scores.append(f1.item())
            except:
                print("Bert Failed")
                bert_f1_scores.append(0.0)
               
            gt_emb = embedder.embed_query(gt)
            # In dtype của pred_emb và gt_emb
            #print(f"pred_emb dtype: {pred_emb.dtype if hasattr(pred_emb, 'dtype') else type(pred_emb)}")
            #print(f"gt_emb dtype: {gt_emb.dtype if hasattr(gt_emb, 'dtype') else type(gt_emb)}")
            # chuyển sang tensor float32
            gt_emb = torch.tensor(gt_emb, dtype=torch.float32)
            # Chuyển đổi dtype sang float32
            if isinstance(pred_emb, np.ndarray):
                pred_emb = torch.from_numpy(pred_emb).float() # Chuyển từ NumPy sang torch.float32
            else:
                pred_emb = pred_emb.float() # Đảm bảo pred_emb là float32
            gt_emb = gt_emb.float() # Đảm bảo gt_emb là float32
            cosine = st_util.cos_sim(pred_emb, gt_emb).item() if pred_emb is not None else 0.0
            cosine_scores.append(cosine)
            # TRIaD
            try:
                q_curr = questions[idx] # Kiểm tra questions[idx] có tồn tại không
                MAX_CHARS = 1200
                safe_pred = (pred or "")[:MAX_CHARS]
                safe_ctx = (m.get("context", "") or "")[:MAX_CHARS]
           
                # Kiểm tra kiểu dữ liệu
                for var, name in [(q_curr, "q_curr"), (safe_pred, "safe_pred"), (gt, "gt"), (safe_ctx, "safe_ctx")]:
                    if not isinstance(var, str):
                        print(f"[ERROR] Expected string for {name}, got {type(var)}: {var}")
                        raise TypeError(f"Invalid type for {name}: {type(var)}")
           
                triad, _ = calculate_triad(safe_pred, q_curr, gt, safe_ctx)
                triad_scores.append(triad)
            except Exception as e:
                print(f"[TRIaD ERROR] {repr(e)}")
                triad_scores.append(0.0)
    # Gộp kết quả vào DataFrame
    df['Prediction'] = predictions
    df['Raw_Respond'] = raw_responses # thêm cột raw respond
    df['EM'] = em_scores
    df['BERT_F1'] = bert_f1_scores
    df['Cosine_Similarity'] = cosine_scores
    df['TRIaD_Score'] = triad_scores
    df['Input_Tokens'] = all_input_tokens
    df['Output_Tokens'] = all_output_tokens
    df['TTFT'] = all_ttft
    df['E2EL'] = all_e2el
    df['Error'] = all_errors
    df['Wrong_Format'] = all_wrong_format
    print('avg....')
    # Tính trung bình
    avg_em = sum(em_scores) / len(em_scores) if em_scores else 0
    avg_bert = sum(bert_f1_scores) / len(bert_f1_scores) if bert_f1_scores else 0
    avg_cosine = sum(cosine_scores) / len(cosine_scores) if cosine_scores else 0
    avg_triad = sum(triad_scores) / len(triad_scores) if triad_scores else 0
    avg_in_tok = sum(all_input_tokens) / len(all_input_tokens) if all_input_tokens else 0
    avg_out_tok = sum(all_output_tokens) / len(all_output_tokens) if all_output_tokens else 0
    avg_ttft = sum(all_ttft) / len(all_ttft) if all_ttft else 0
    avg_e2el = sum(all_e2el) / len(all_e2el) if all_e2el else 0
    err_rate = sum(all_errors) / len(all_errors) if all_errors else 0
    wrong_rate = sum(all_wrong_format) / len(all_wrong_format) if all_wrong_format else 0
   
    # Logging
    logging.info("\n=== Average Evaluation Scores ===")
    logging.info(f"EM: {avg_em:.4f}")
    logging.info(f"BERT-Score F1: {avg_bert:.4f}")
    logging.info(f"Cosine Similarity: {avg_cosine:.4f}")
    logging.info(f"TRIaD Score: {avg_triad:.4f}")
    logging.info("\n=== Average Performance Metrics ===")
    logging.info(f"Input Tokens: {avg_in_tok:.2f}")
    logging.info(f"Output Tokens: {avg_out_tok:.2f}")
    logging.info(f"TTFT: {avg_ttft:.2f} sec")
    logging.info(f"E2EL: {avg_e2el:.2f} sec")
    logging.info(f"Error Rate: {err_rate:.2%}")
    logging.info(f"Wrong Format Rate: {wrong_rate:.2%}")
    print("out...")
    # Xuất kết quả JSON với các cột cần thiết
    output_path = "/kaggle/working/Pipeline Testing LLM 1.json"
    # Đổi tên cột Raw_Respond -> respond
    df.rename(columns={"Raw_Respond": "Respond"}, inplace=True)
    # Giữ lại các cột mong muốn
    cols_to_keep = [
        "Disease", "Question", "Respond",
        "EM", "BERT_F1", "Cosine_Similarity", "TRIaD_Score",
        "Input_Tokens", "Output_Tokens", "TTFT", "E2EL", "Error", "Wrong_Format"
    ]
    df_final = df[cols_to_keep]
    # Xuất ra file JSON
    df_final.to_json(output_path, orient="records", indent=4, force_ascii=False)
    logging.info(f"Results saved to {output_path}")
    return df

In [ ]:
reranker = load_viranker()
llm = load_llm()
csv_file = '/kaggle/input/testdatasetllm/ViMedical_Disease_Unique_1.csv'
results = evaluate_csv(csv_file, llm, embedding_generator, reranker)

In [ ]:
results